# CaptionLab controlled evaluation

This notebook evaluates the three completed runs on the frozen 810-image test split. Greedy decoding is the primary protocol. It exports every prediction and reference; beam search is not used to choose the reported winner.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess, torch
REPO = Path('/content/CaptionLab')
DRIVE_ROOT = Path('/content/drive/MyDrive/CaptionLab')
EVALUATION_COMMIT = 'ed94547cfa1d60baf91c1e76d7687952276541fc'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', 'https://github.com/SahilBh01r1769/image-captioning.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', EVALUATION_COMMIT], cwd=REPO, check=True)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements-eval.txt', 'kagglehub'], cwd=REPO, check=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Pinned evaluation code:', EVALUATION_COMMIT, '| device:', device)

## Verify the frozen artifacts

This uses the checkpoints, captions backup, split, vocabulary, and feature cache already in Drive. It does not retrain or rebuild any identity-bearing artifact.

In [ ]:
RUNS = DRIVE_ROOT / 'runs'
VOCAB = DRIVE_ROOT / 'models' / 'vocabulary.pkl'
SPLIT = DRIVE_ROOT / 'splits' / 'flickr8k_seed42.json'
CAPTIONS = DRIVE_ROOT / 'dataset_metadata' / 'captions.txt'
DRIVE_CACHE = DRIVE_ROOT / 'feature_cache' / 'flickr8k_resnet50_spatial.pt'
required = [VOCAB, SPLIT, CAPTIONS, DRIVE_CACHE] + [RUNS / name / 'checkpoints' / 'best.pt' for name in ('baseline_seed42','attention_seed42','attention_coverage_seed42')]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, 'Missing required artifacts:\n' + '\n'.join(missing)
(REPO / 'data' / 'Flickr8k').mkdir(parents=True, exist_ok=True)
shutil.copy2(CAPTIONS, REPO / 'data' / 'Flickr8k' / 'captions.txt')
(REPO / 'models').mkdir(exist_ok=True); shutil.copy2(VOCAB, REPO / 'models' / 'vocabulary.pkl')
(REPO / 'splits').mkdir(exist_ok=True); shutil.copy2(SPLIT, REPO / 'splits' / 'flickr8k_seed42.json')
LOCAL_CACHE = Path('/content/flickr8k_resnet50_spatial.pt')
if not LOCAL_CACHE.exists(): shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
print('All three best checkpoints and protocol artifacts are present.')

## Smoke-check three images

This checks checkpoint loading, identity validation, and cached-feature decoding. Corpus metrics are skipped here so a three-image pipeline check cannot be mistaken for evidence or delayed by metric startup.

In [ ]:
SMOKE = Path('/content/captionlab_evaluation_smoke')
if SMOKE.exists(): shutil.rmtree(SMOKE)
subprocess.run(['python','evaluate.py','--runs_dir',str(RUNS),'--feature_cache',str(LOCAL_CACHE),'--vocabulary',str(VOCAB),'--output_dir',str(SMOKE),'--max_images','3','--skip_metrics'], cwd=REPO, check=True)
print('Smoke status:', __import__('json').loads((SMOKE/'comparison.json').read_text())['status'])

## Evaluate all 810 test images

The output is built in temporary Colab storage and copied to Drive only after all three runs finish. If this cell disconnects, rerun it; the temporary directory is discarded so partial files cannot be mistaken for a completed evaluation.

In [ ]:
LOCAL_EVAL = Path('/content/captionlab_evaluation_seed42_greedy')
DRIVE_EVAL = DRIVE_ROOT / 'evaluation_seed42_greedy'
assert not DRIVE_EVAL.exists(), f'Refusing to overwrite completed evidence: {DRIVE_EVAL}'
if LOCAL_EVAL.exists(): shutil.rmtree(LOCAL_EVAL)
subprocess.run(['python','evaluate.py','--runs_dir',str(RUNS),'--feature_cache',str(LOCAL_CACHE),'--vocabulary',str(VOCAB),'--output_dir',str(LOCAL_EVAL)], cwd=REPO, check=True)
subprocess.run(['python','report_evaluation.py','--evaluation_dir',str(LOCAL_EVAL),'--runs_dir',str(RUNS)], cwd=REPO, check=True)
shutil.copytree(LOCAL_EVAL, DRIVE_EVAL)
print((DRIVE_EVAL / 'comparison.md').read_text())

## Optional fixed-sample attention overlays

Run this if you want image overlays. KaggleHub downloads Flickr8k directly to Colab; no API key or device upload is needed. The six examples are evenly spaced through the frozen test order, not selected for looking good.

In [ ]:
import kagglehub
download_root = Path(kagglehub.dataset_download('adityajn105/flickr8k'))
image_dirs = [path for path in download_root.rglob('Images') if path.is_dir()]
assert len(image_dirs) == 1, f'Expected one Images directory, found {len(image_dirs)}'
subprocess.run(['python','report_evaluation.py','--evaluation_dir',str(DRIVE_EVAL),'--runs_dir',str(RUNS),'--images_dir',str(image_dirs[0]),'--attention_examples','6'], cwd=REPO, check=True)
print('Attention overlays:', DRIVE_EVAL / 'attention_examples')

## Package the evidence to return

The ZIP includes aggregate metrics, every prediction with all references, raw token probabilities/attention weights, loss curves, optional overlays, and an intentionally blank failure-gallery worksheet.

In [ ]:
archive_base = DRIVE_ROOT / 'CaptionLab_evaluation_bundle'
archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=DRIVE_EVAL))
print('Return this file:', archive, f'({archive.stat().st_size / 2**20:.1f} MiB)')